# 01. Data collection

Phase 1, task 1.4.3. This notebook records the real queries used to find and pull data for Tashkent, and what each pull returned. It does not analyse the data. Quality checks are in `02_data_quality_assessment.ipynb`, and sources and licences are in `docs/datasets/data_sources.md`.

**How to run.** Run from the repository root or from `notebooks/` after `python scripts/download_data.py` has created `data/raw/` (see the commands below; settings are in `configs/sources.toml`). Cells that query live services are skipped when `AERON_OFFLINE=1`, and the OpenAQ cell also needs `OPENAQ_API_KEY` (environment or `.env`). Live results change over time, so numbers in this notebook are from the retrieval date stored in the metadata files.

| Source | Command that downloads it |
|---|---|
| OpenAQ hourly series | `python scripts/download_data.py --only openaq` |
| Open-Meteo ERA5 weather (core and extra variables) | `python scripts/download_data.py --only era5` |
| Open-Meteo archived forecasts | `python scripts/download_data.py --only forecast` |
| Mendeley TIIAME dataset | manual download, see section 5 |

In [1]:
import json
import os
import ssl
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
OFFLINE = os.environ.get("AERON_OFFLINE", "0") == "1"
CONTEXT = ssl.create_default_context(
    cafile="/etc/ssl/cert.pem" if Path("/etc/ssl/cert.pem").exists() else None
)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("offline:", OFFLINE)


def api_key():
    key = os.environ.get("OPENAQ_API_KEY")
    env_file = ROOT / ".env"
    if not key and env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith("OPENAQ_API_KEY="):
                key = line.split("=", 1)[1].strip()
    return key


def get_json(url, headers=None, timeout=60):
    req = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(req, timeout=timeout, context=CONTEXT) as resp:
        return json.load(resp)

offline: False


## 1. OpenAQ: locations around Tashkent

Query: all locations within 25 km of the city centre (41.3111 N, 69.2797 E). This is the query that found the two PM2.5 locations used in the project.

In [2]:
key = api_key()
if OFFLINE or not key:
    print("skipped: offline or no OPENAQ_API_KEY")
else:
    query = urllib.parse.urlencode(
        {"coordinates": "41.3111,69.2797", "radius": 25000, "limit": 100}
    )
    result = get_json(
        f"https://api.openaq.org/v3/locations?{query}", {"X-API-Key": key}
    )
    rows = []
    for loc in result["results"]:
        names = [s["parameter"]["name"] for s in loc["sensors"]]
        rows.append(
            {
                "id": loc["id"],
                "name": loc["name"],
                "provider": (loc.get("provider") or {}).get("name"),
                "monitor": loc.get("isMonitor"),
                "lat": loc["coordinates"]["latitude"],
                "lon": loc["coordinates"]["longitude"],
                "first_utc": (loc["datetimeFirst"] or {}).get("utc"),
                "last_utc": (loc["datetimeLast"] or {}).get("utc"),
                "parameters": ", ".join(sorted(set(names))),
            }
        )
    print("locations found:", result["meta"]["found"])
    print(pd.DataFrame(rows).to_string(index=False))

locations found: 2
     id                         name          provider  monitor       lat       lon            first_utc             last_utc                                      parameters
   8881 US Diplomatic Post: Tashkent StateAir Tashkent     True 41.367200 69.272500 2018-11-27T22:00:00Z 2025-03-04T12:00:00Z                                        o3, pm25
4902926                    Sputnik-4       AirGradient    False 41.204655 69.232522 2025-06-29T19:00:00Z 2026-01-19T17:00:00Z pm1, pm25, relativehumidity, temperature, um003


## 2. OpenAQ: what was downloaded

`aeron.ingest.openaq` pulls one month per request (wider ranges time out), caches each month under `data/raw/openaq/_months/`, writes `data/raw/openaq/<name>_hourly.csv` and records the retrieval time and sensor ids in `metadata.json`. Finished months are skipped on later runs.

In [3]:
meta = json.loads((RAW / "openaq" / "metadata.json").read_text())
print("retrieved (UTC):", meta["retrieved_utc"], "|", meta["source"])
files = []
for name, info in meta["files"].items():
    df = pd.read_csv(
        RAW / "openaq" / f"{name}_hourly.csv", parse_dates=["datetime_utc"]
    )
    files.append(
        {
            "file": name,
            "location_id": info["location_id"],
            "sensor_id": info["sensor_id"],
            "parameter": info["parameter"],
            "rows": len(df),
            "empty_values": int(df["value"].isna().sum()),
            "first_utc": df["datetime_utc"].min(),
            "last_utc": df["datetime_utc"].max(),
        }
    )
pd.DataFrame(files)

retrieved (UTC): 2026-11-17T11:32:31+00:00 | OpenAQ API v3, /sensors/{id}/hours


,file,location_id,sensor_id,parameter,rows,empty_values,first_utc,last_utc
0,us_embassy_pm25,8881,25916,pm25,51492,7411,2018-11-27 21:00:00+00:00,2025-03-04 11:00:00+00:00
1,sputnik4_pm25,4902926,13465748,pm25,3277,0,2025-06-29 18:00:00+00:00,2026-01-19 16:00:00+00:00
2,sputnik4_humidity,4902926,13465749,relativehumidity,3277,0,2025-06-29 18:00:00+00:00,2026-01-19 16:00:00+00:00
3,sputnik4_temperature,4902926,13465750,temperature,3277,0,2025-06-29 18:00:00+00:00,2026-01-19 16:00:00+00:00


In [4]:
pd.read_csv(RAW / "openaq" / "us_embassy_pm25_hourly.csv").head()

,datetime_utc,value,percent_complete,observed,expected
0,2018-11-27T21:00:00Z,21.0,100.0,1,1
1,2018-11-27T22:00:00Z,36.0,100.0,1,1
2,2018-11-27T23:00:00Z,51.0,100.0,1,1
3,2018-11-28T00:00:00Z,36.0,100.0,1,1
4,2018-11-28T01:00:00Z,33.0,100.0,1,1


## 3. Open-Meteo: weather and archived forecasts

Three downloads share the same location (the US Diplomatic Post coordinates). Request parameters and retrieval times are in the metadata files.

In [5]:
rows = []
for fname in (
    "metadata.json",
    "era5_extra_metadata.json",
    "forecast_previous_runs_metadata.json",
):
    m = json.loads((RAW / "openmeteo" / fname).read_text())
    rows.append(
        {
            "metadata file": fname,
            "source": m["source"],
            "retrieved_utc": m["retrieved_utc"],
            "start": m["start"],
            "end": m["end"],
            "rows": m["rows"],
            "variables": len(m["variables"]),
        }
    )
pd.DataFrame(rows)

,metadata file,source,retrieved_utc,start,end,rows,variables
0,metadata.json,Open-Meteo historical weather API (ERA5 reanal...,2026-11-17T11:32:37+00:00,2018-11-01,2026-11-10,70368,9
1,era5_extra_metadata.json,Open-Meteo historical weather API (ERA5 reanal...,2026-11-17T11:32:37+00:00,2018-11-01,2026-11-10,70368,5
2,forecast_previous_runs_metadata.json,Open-Meteo Previous Runs API,2026-11-17T10:00:21+00:00,2024-01-19,2025-03-01,9792,11


In [6]:
for fname in (
    "era5_tashkent_hourly.csv",
    "era5_extra_tashkent_hourly.csv",
    "forecast_previous_runs_hourly.csv",
):
    df = pd.read_csv(RAW / "openmeteo" / fname, parse_dates=["datetime_utc"])
    print(
        f"{fname}: {len(df)} rows, {df.shape[1] - 1} variables,",
        f"{df['datetime_utc'].min()} to {df['datetime_utc'].max()}",
    )

era5_tashkent_hourly.csv: 70368 rows, 9 variables, 2018-11-01 00:00:00+00:00 to 2026-11-10 23:00:00+00:00
era5_extra_tashkent_hourly.csv: 70368 rows, 5 variables, 2018-11-01 00:00:00+00:00 to 2026-11-10 23:00:00+00:00
forecast_previous_runs_hourly.csv: 9792 rows, 33 variables, 2024-01-19 00:00:00+00:00 to 2025-03-01 23:00:00+00:00


Which weather variables exist as archived forecasts matters for evaluation (see `docs/research/methodology.md`). This live check asks the Previous Runs API for the forecast issued two days earlier, over three winter days in 2024, and reports how many hours are non-empty.

In [7]:
if OFFLINE:
    print("skipped: offline")
else:
    variables = [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "boundary_layer_height",
        "snow_depth",
        "shortwave_radiation",
        "pressure_msl",
    ]
    columns = ",".join(f"{v}_previous_day2" for v in variables)
    query = urllib.parse.urlencode(
        {
            "latitude": 41.3672,
            "longitude": 69.2725,
            "timezone": "UTC",
            "start_date": "2024-02-01",
            "end_date": "2024-02-03",
            "hourly": columns,
        }
    )
    try:
        data = get_json(f"https://previous-runs-api.open-meteo.com/v1/forecast?{query}")
        hourly = data["hourly"]
        table = pd.DataFrame(
            {
                "variable": variables,
                "non-empty hours": [
                    sum(x is not None for x in hourly[f"{v}_previous_day2"])
                    for v in variables
                ],
                "of": len(hourly["time"]),
            }
        )
        print(table.to_string(index=False))
    except urllib.error.URLError as err:
        print("request failed:", err)

             variable  non-empty hours  of
       temperature_2m               72  72
 relative_humidity_2m               72  72
       wind_speed_10m               72  72
boundary_layer_height                0  72
           snow_depth                0  72
  shortwave_radiation               72  72
         pressure_msl               72  72


## 4. Tashkent Open Data portal

The portal publishes air-quality datasets 133 and 230. The cell reads their metadata and tries the download endpoint listed there. At the retrieval date (2026-11-17) the download returned HTTP 500. If it works now, the first download is the starting point for the multi-station question (not required: the project accepts a single-station study).

In [8]:
if OFFLINE:
    print("skipped: offline")
else:
    base = "https://opendata-back.tashkent.uz/en/api/data/all"
    rows = []
    for dataset in (133, 230):
        info = get_json(f"{base}/{dataset}/")
        try:
            with urllib.request.urlopen(
                f"{base}/{dataset}/download/", timeout=60, context=CONTEXT
            ) as resp:
                status = resp.status
        except urllib.error.HTTPError as err:
            status = err.code
        except urllib.error.URLError as err:
            status = str(err.reason)
        rows.append(
            {
                "dataset": dataset,
                "name": info["name_en"],
                "owner": info["organisation"]["name_en"],
                "first_publish": info["first_publish_date"],
                "last_publish": info["last_publish_date"],
                "formats": ", ".join(info["formats"]),
                "download status": status,
            }
        )
    print(pd.DataFrame(rows).to_string(index=False))

 dataset                               name                                                                   owner first_publish last_publish              formats  download status
     133    Tashkent Air Quality Monitoring Department of Digital Development under the Khokimiyat of Tashkent city    2024-03-18   2026-02-12 xlsx, csv, xls, json              500
     230 Dynamics of changes in air quality Department of Digital Development under the Khokimiyat of Tashkent city    2025-02-20   2025-02-20 xlsx, csv, xls, json              500


## 5. Mendeley Data (TIIAME station)

Manual download of `all_sensor.csv` and `README.md` from https://data.mendeley.com/datasets/nmpz7jds4f/2 (CC BY 4.0) into `data/raw/mendeley_tashkent/`. The check below confirms the file matches the hash published by Mendeley.

In [9]:
import hashlib

path = RAW / "mendeley_tashkent" / "all_sensor.csv"
published = "15c334cd5ba15bf96c35ea30a89d1a2edd41d6b2244d686e93c6f4a37cad8b37"
digest = hashlib.sha256(path.read_bytes()).hexdigest()
print("sha256 matches published hash:", digest == published)
df = pd.read_csv(path)
print(len(df), "rows, columns:", list(df.columns))
df.head(3)

sha256 matches published hash: True
9546 rows, columns: ['Date', 'Time', 'CO/ppm', 'NO/ppb', 'O3/ppb', 'PM10/ug/m3', 'PM25/ug/m3', 'SO2/ppb', 'Temperature', 'tsp/ug/m3', 'tvoc/ppm', 'wind_speed/m/s', 'Humidity/%RH', 'wind_degree/Ўг']


,Date,Time,CO/ppm,NO/ppb,O3/ppb,PM10/ug/m3,PM25/ug/m3,SO2/ppb,Temperature,tsp/ug/m3,tvoc/ppm,wind_speed/m/s,Humidity/%RH,wind_degree/Ўг
0,2/26/2024,11:22:19,4.90,138.69,13.11,21.0,21.0,64.0,4.1,21.0,0.00,0.4,82.0,103.0
1,2/26/2024,11:52:19,3.92,0.75,1.73,12.0,12.0,65.0,4.7,12.0,0.01,0.0,79.0,141.0
2,2/26/2024,12:22:19,3.48,16.13,13.79,23.0,23.0,63.0,4.6,23.0,0.00,0.9,78.0,145.0


## 6. Summary of pulls

All values below are computed from the files above.

In [10]:
summary = []
for name in meta["files"]:
    df = pd.read_csv(
        RAW / "openaq" / f"{name}_hourly.csv", parse_dates=["datetime_utc"]
    )
    valid = df.dropna(subset=["value"])
    span_hours = (
        int(
            (valid["datetime_utc"].max() - valid["datetime_utc"].min()).total_seconds()
            // 3600
        )
        + 1
    )
    summary.append(
        {
            "series": name,
            "valid hours": len(valid),
            "span hours": span_hours,
            "valid share of span": round(len(valid) / span_hours, 3),
            "first": valid["datetime_utc"].min().date(),
            "last": valid["datetime_utc"].max().date(),
        }
    )
pd.DataFrame(summary)

,series,valid hours,span hours,valid share of span,first,last
0,us_embassy_pm25,44081,54765,0.805,2018-11-27,2025-02-25
1,sputnik4_pm25,3277,4895,0.669,2025-06-29,2026-01-19
2,sputnik4_humidity,3277,4895,0.669,2025-06-29,2026-01-19
3,sputnik4_temperature,3277,4895,0.669,2025-06-29,2026-01-19
